# YOLOv8s + SPDConv + Inner-IoU Colab Experiment

This notebook clones the `yolov8s_kuangjing` repository, installs the local Ultralytics package from source, enables `Inner-IoU`, and trains `YOLOv8s` with `SPDConv` only in the first backbone downsampling layer.

For your real experiment, only change `DATA_CFG`, `EPOCHS`, `BATCH`, and `INNER_RATIO`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/content/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .

In [ ]:
from pathlib import Path
import requests
from PIL import Image

assets_dir = Path("/content/yolov8s_kuangjing/ultralytics/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)

    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / "bus.jpg", "https://ultralytics.com/images/bus.jpg")
ensure_valid_image(assets_dir / "zidane.jpg", "https://ultralytics.com/images/zidane.jpg")
print("assets fixed")

In [ ]:
import os
from pathlib import Path

MODEL_CFG = "ultralytics/cfg/models/v8/yolov8s_spdconv_inneriou.yaml"
DATA_CFG = "ultralytics/cfg/datasets/coco8.yaml"  # Replace with your dataset yaml
EPOCHS = 40
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = 0
PROJECT = "runs/colab"
NAME = "yolov8s_spdconv_inneriou_colab"
SEED = 42
INNER_RATIO = 0.7

os.environ["ULTRALYTICS_CLS_LOSS"] = "bce"
os.environ["ULTRALYTICS_IOU_LOSS"] = "inner_iou"
os.environ["ULTRALYTICS_INNER_IOU_RATIO"] = str(INNER_RATIO)

assert Path(MODEL_CFG).exists(), f"Missing model config: {MODEL_CFG}"
assert Path(DATA_CFG).exists(), f"Missing data config: {DATA_CFG}"
assert Path("ultralytics/nn/modules/block.py").exists(), "Missing ultralytics/nn/modules/block.py"
assert Path("ultralytics/utils/loss.py").exists(), "Missing ultralytics/utils/loss.py"
assert Path("ultralytics/utils/metrics.py").exists(), "Missing ultralytics/utils/metrics.py"

print("MODEL_CFG:", MODEL_CFG)
print("DATA_CFG:", DATA_CFG)
print("INNER_RATIO:", INNER_RATIO)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_CFG)
results = model.train(
    data=DATA_CFG,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT,
    name=NAME,
    pretrained="yolov8s.pt",
    cache=False,
    amp=True,
    patience=50,
    seed=SEED,
    deterministic=True,
)
results

In [ ]:
from pathlib import Path

run_dir = Path(PROJECT) / NAME
print("Run directory:", run_dir)
print("results.csv exists:", (run_dir / "results.csv").exists())
print("best.pt exists:", (run_dir / "weights" / "best.pt").exists())
print("last.pt exists:", (run_dir / "weights" / "last.pt").exists())